# GameTheory 16j — Screening et loyer d'information

**Navigation** : [<< 16e-Players-LLM](GameTheory-16e-LLM-Players-Othman-Sandholm-Python.ipynb) | [Index GameTheory](README.md) | [Capstone 21 -- Choix social >>](GameTheory-21-Capstone-Choix-Social-Python.ipynb)

**Le coût mesuré de l'asymétrie d'information** dans le problème classique de *screening*
à sélection adverse (Maskin–Riley, Myerson).

À la famille **16 (Mécanismes)**, à côté de 16i (Appariage stable) : un principal
(employeur) offre un contrat à un agent (salarié) dont il **ne connaît pas le type**.
On mesure trois choses :

1. le **loyer d'information** que le principal doit verser au type haut pour que les deux
   types s'autosélectionnent honnêtement ;
2. la **distorsion** du contrat du type bas (on le fait travailler *moins* que l'optimum) —
   le type haut, lui, reste à son **premier optimum** (théorème de non-distorsion au
   sommet) ;
3. le **coût total de l'asymétrie** = surplus avec information complète − surplus avec
   screening, et sa **décomposition** en loyer + perte d'efficacité.

Toutes les valeurs sont **calculées** dans le carnet (numpy, CPU seul), pas énoncées.


## 1. Le modèle

Un salarié est de l'un de deux **types** (productivités) :

- type **bas** $\theta_L = 1$, type **haut** $\theta_H = 2$ ;
- probabilité $p$ que le salarié soit du type haut.

Le principal propose un contrat $(a, t)$ : un niveau d'effort $a$ et une rémunération $t$.

| | Formule | Rôle |
|---|---|---|
| Production | $V(\theta, a) = \theta \, a$ | ce que l'effort $a$ d'un type $\theta$ rapporte au principal |
| Coût d'effort | $C(\theta, a) = \dfrac{a^2}{2\theta}$ | ce que l'effort $a$ coûte au type $\theta$ |

Le type haut trouve l'effort **moins coûteux** ($C$ décroît en $\theta$) : c'est la condition
de **croisement simple** (single-crossing) qui rend le screening possible. L'agent a une
utilité $U(\theta) = t - C(\theta, a)$ ; le principal un surplus $S(\theta) = V(\theta, a) - t$.

> Le type est **non observable** par le principal : c'est l'asymétrie d'information.


In [1]:
import numpy as np

# --- Types et probabilité ---
theta_L = 1.0
theta_H = 2.0
p_H     = 0.5     # probabilité que le salarié soit du type haut

# --- Production et coût d'effort ---
def V(th, a):  return th * a
def C(th, a):  return a**2 / (2.0 * th)

print(f"theta_L = {theta_L}, theta_H = {theta_H}, p_H = {p_H}")
print(f"coût d'effort du type haut  C(2, a=2) = {C(theta_H, 2.0)}")
print(f"coût d'effort du type bas   C(1, a=2) = {C(theta_L, 2.0)}")
print("(le type haut trouve l'effort moins cher : croisement simple)")

theta_L = 1.0, theta_H = 2.0, p_H = 0.5
coût d'effort du type haut  C(2, a=2) = 1.0
coût d'effort du type bas   C(1, a=2) = 2.0
(le type haut trouve l'effort moins cher : croisement simple)


### Lecture

À effort égal ($a=2$), le type haut paie un coût $1$ là où le type bas en paie $2$.
C'est cette **asymétrie de coût** que le mécanisme va exploiter : en proposant le bon
contrat, on fait en sorte que chaque type *choisisse de lui-même* le sien.


## 2. Premier optimum — le principal connaît le type (benchmark)

Sans asymétrie, le principal fixe pour **chaque** type l'effort qui maximise son surplus
$S(\theta) = \theta a - a^2/(2\theta)$, et paie juste le coût (l'agent ne garde **aucun
loyer** : l'agent de participation est liant, $U=0$).

Le maximum en $a$ donne $a^*(\theta) = \theta^2$ (dérivée nulle : $\theta - a/\theta = 0$).


In [2]:
# --- Premier optimum (information complète) ---
a_L_1F = theta_L**2
a_H_1F = theta_H**2
t_L_1F = C(theta_L, a_L_1F)          # loyer = coût  ->  loyer d'information = 0
t_H_1F = C(theta_H, a_H_1F)
S_L_1F = V(theta_L, a_L_1F) - t_L_1F
S_H_1F = V(theta_H, a_H_1F) - t_H_1F

print("PREMIER OPTIMUM (le principal voit le type) :")
print(f"  type bas : a* = {a_L_1F:.2f},  t = {t_L_1F:.3f},  surplus = {S_L_1F:.3f}")
print(f"  type haut: a* = {a_H_1F:.2f},  t = {t_H_1F:.3f},  surplus = {S_H_1F:.3f}")
print(f"  (aucun loyer d'information : l'agent est payé à juste son coût)")

PREMIER OPTIMUM (le principal voit le type) :
  type bas : a* = 1.00,  t = 0.500,  surplus = 0.500
  type haut: a* = 4.00,  t = 4.000,  surplus = 4.000
  (aucun loyer d'information : l'agent est payé à juste son coût)


### Lecture

Avec information complète, le type haut travaille à $a^* = 4$ et le principal en retire
un surplus de **4.00**. C'est le **plafond** : tout ce qui manque plus bas est dû à
l'asymétrie. On garde ce chiffre en tête.


## 3. Second optimum — le principal ne voit pas le type (screening)

Le principal propose un **menu** de deux contrats. Les deux contraintes d'un menu
séparateur optimal :

- **Participation** du type bas liante : $U_L = 0$ (on ne paie pas de loyer au type bas) ;
- **Auto-sélection du type haut** liante : le type haut ne doit pas préférer le contrat
  du type bas — c'est cette contrainte qui crée le loyer.

Le résultat classique (croisement simple) est le **théorème de non-distorsion au
sommet** : **le type haut reçoit l'effort du premier optimum** ($a_H = \theta_H^2$,
non déformé), et **le type bas voit son effort abaissé** ($a_L < \theta_L^2$). Le loyer
d'information du type haut est alors une **fonction du contrat du type bas** :
$$U_H = C(\theta_L, a_L) - C(\theta_H, a_L) .$$

En fixant $a_H = \theta_H^2$ et en exprimant le surplus espéré du principal, son problème
se réduit au choix unique de $a_L$ :
$$\max_{a_L}\; p_H\,S_H(a_L) + (1-p_H)\,S_L(a_L)
\quad\Rightarrow\quad
a_L^{2B} = \frac{(1-p_H)\,\theta_L}{p_H\left(\frac{1}{\theta_L}-\frac{1}{\theta_H}\right) + \frac{1-p_H}{\theta_L}} .$$


In [3]:
# --- Second optimum (screening) — théorème de non-distorsion au sommet ---
# Le type HAUT n'est PAS déformé : il travaille à son premier optimum.
a_H_2B = theta_H**2                     # type haut : NON déformé (= 1er optimum)
# Le type BAS est déformé : a_L résout le FOC du surplus espéré du principal.
a_L_2B = ((1 - p_H) * theta_L) / (p_H * (1/theta_L - 1/theta_H) + (1 - p_H)/theta_L)

# Loyer d'information du type haut = fonction du contrat du type BAS
U_H = C(theta_L, a_L_2B) - C(theta_H, a_L_2B)

t_L_2B = C(theta_L, a_L_2B)             # U_L = 0 (participation liante)
t_H_2B = C(theta_H, a_H_2B) + U_H       # t_H = coût haut + loyer (auto-sélection liante)
S_L_2B = V(theta_L, a_L_2B) - t_L_2B
S_H_2B = V(theta_H, a_H_2B) - t_H_2B

print("SECOND OPTIMUM (screening, le type est caché) :")
print(f"  type haut : a = {a_H_2B:.2f}  (NON déformé = 1er optimum),  surplus = {S_H_2B:.3f}")
print(f"  type bas  : a = {a_L_2B:.3f}  (abaissé vs {a_L_1F:.2f}),   surplus = {S_L_2B:.3f}")
print(f"  LOYER D'INFORMATION du type haut  U_H = {U_H:.3f}  (= fonction de a_L)")
print(f"  rémunération type haut t_H = {t_H_2B:.3f}  (= coût {C(theta_H, a_H_2B):.3f} + loyer {U_H:.3f})")


SECOND OPTIMUM (screening, le type est caché) :
  type haut : a = 4.00  (NON déformé = 1er optimum),  surplus = 3.889
  type bas  : a = 0.667  (abaissé vs 1.00),   surplus = 0.444
  LOYER D'INFORMATION du type haut  U_H = 0.111  (= fonction de a_L)
  rémunération type haut t_H = 4.111  (= coût 4.000 + loyer 0.111)


### Lecture

Contre-intuitif : c'est le **type bas** qui est déformé (de $a^* = 1$ à $a_L = 0.67$),
pas le type haut. Le type haut travaille à son **premier optimum** $a_H = 4$ — c'est le
**théorème de non-distorsion au sommet**. En échange de ce contrat non déformé, le
principal lui verse un **loyer d'information** $U_H = 0.11$ : c'est le prix pour qu'il
ne se fasse pas passer pour le type bas. Ce loyer est une fonction du contrat du type
**bas** — c'est la distorsion de $a_L$ qui le finance.


## 4. Le coût de l'asymétrie, et sa décomposition

Le **coût de l'asymétrie d'information** est la perte de surplus par rapport au premier
optimum (§2), et il touche **les deux types** :

- le **type haut** perd son **loyer d'information** $U_H$ (son effort n'est PAS déformé) ;
- le **type bas** subit la **distorsion** de son effort ($a_L$ abaissé de $1$ à $0.67$).

C'est l'image miroir de l'intuition naïve : dans le screening à croisement simple,
**toute la distorsion est en bas, tout le loyer est en haut**.


In [4]:
# --- Coût de l'asymétrie + décomposition (les deux types) ---
# Type haut : tout le coût est le loyer (son effort n'est pas déformé)
cost_H   = S_H_1F - S_H_2B
rent_H   = U_H
distort_H = 0.0                                # a_H non déformé -> perte d'efficacité nulle
# Type bas : tout le coût est la distorsion (pas de loyer, U_L = 0)
cost_L   = S_L_1F - S_L_2B
distort_L = S_L_1F - (V(theta_L, a_L_2B) - C(theta_L, a_L_2B))
rent_L    = 0.0                                # U_L = 0

print("COÛT DE L'ASYMÉTRIE (par salarié, par type) :")
print(f"  type haut : coût = {cost_H:.3f}  = loyer {rent_H:.3f} + distorsion {distort_H:.3f}")
print(f"  type bas  : coût = {cost_L:.3f}  = loyer {rent_L:.3f} + distorsion {distort_L:.3f}")
assert abs(rent_H + distort_H - cost_H) < 1e-9, "décomp. type haut incohérente"
assert abs(rent_L + distort_L - cost_L) < 1e-9, "décomp. type bas incohérente"

# Coût espéré (moyenne sur les deux types) + décomposition espérée
cost_expected    = p_H * cost_H + (1 - p_H) * cost_L
rent_expected    = p_H * rent_H + (1 - p_H) * rent_L
distort_expected = p_H * distort_H + (1 - p_H) * distort_L
print(f"  coût ESPÉRÉ (p_H = {p_H}) = {cost_expected:.3f}")
print(f"    dont loyer d'info espéré   = {rent_expected:.3f}   ({100*rent_expected/cost_expected:.0f} %)")
print(f"    dont distorsion espérée    = {distort_expected:.3f}   ({100*distort_expected/cost_expected:.0f} %)")


COÛT DE L'ASYMÉTRIE (par salarié, par type) :
  type haut : coût = 0.111  = loyer 0.111 + distorsion 0.000
  type bas  : coût = 0.056  = loyer 0.000 + distorsion 0.056
  coût ESPÉRÉ (p_H = 0.5) = 0.083
    dont loyer d'info espéré   = 0.056   (67 %)
    dont distorsion espérée    = 0.028   (33 %)


### Lecture — le résultat central

Avec $\theta_L=1, \theta_H=2, p_H=0.5$, le coût **espéré** de l'asymétrie est
$1/12 \approx 0.083$. Il se décompose en **loyer d'information** ($1/18$, porté à 100 %
par le type haut) + **distorsion** ($1/36$, portée à 100 % par le type bas). Le partage
n'est PAS 50/50 entre loyer et distorsion — c'est une conséquence des paramètres.
L'essentiel : **le type haut paie en loyer sans être déformé ; le type bas est déformé
sans loyer**. C'est la signature du screening à croisement simple.


## 5. Sensibilité — le coût croît avec l'écart de type

Que se passe-t-il quand les deux types sont **plus différents** ? On fixe
$\theta_L = 1$ et on fait varier $\theta_H$, en recalculant le coût **espéré** de
l'asymétrie (loyer du type haut + distorsion du type bas) pour chaque écart.


In [5]:
# --- Sensibilité : coût d'asymétrie espéré en fonction de l'écart de type ---
print(f"{'theta_H':>8} | {'a_H':>6} | {'a_L 2B':>8} | {'loyer H':>9} | {'dist. L':>9} | {'coût esp.':>10}")
print("-" * 60)
for thH in [1.25, 1.5, 2.0, 2.5, 3.0]:
    aH = thH**2                                        # type haut : 1er optimum (non déformé)
    aL = ((1 - p_H) * theta_L) / (p_H * (1/theta_L - 1/thH) + (1 - p_H)/theta_L)  # type bas : déformé
    rentH = C(theta_L, aL) - C(thH, aL)               # loyer du type haut = f(a_L)
    S1F_H = V(thH, aH) - C(thH, aH)
    S2B_H = V(thH, aH) - (C(thH, aH) + rentH)
    costH = S1F_H - S2B_H                              # = rentH (a_H non déformé)
    aL1F  = theta_L**2
    S1F_L = V(theta_L, aL1F) - C(theta_L, aL1F)
    S2B_L = V(theta_L, aL) - C(theta_L, aL)
    distL = S1F_L - S2B_L                              # distorsion du type bas
    cost_exp = p_H * costH + (1 - p_H) * distL
    print(f"{thH:>8.2f} | {aH:>6.2f} | {aL:>8.3f} | {rentH:>9.3f} | {distL:>9.3f} | {cost_exp:>10.3f}")


 theta_H |    a_H |   a_L 2B |   loyer H |   dist. L |  coût esp.
------------------------------------------------------------
    1.25 |   1.56 |    0.833 |     0.069 |     0.014 |      0.042
    1.50 |   2.25 |    0.750 |     0.094 |     0.031 |      0.063
    2.00 |   4.00 |    0.667 |     0.111 |     0.056 |      0.083
    2.50 |   6.25 |    0.625 |     0.117 |     0.070 |      0.094
    3.00 |   9.00 |    0.600 |     0.120 |     0.080 |      0.100


### Lecture

Le coût de l'asymétrie **croît de façon convexe** avec l'écart de type : $\approx 0.04$
pour $\theta_H=1.25$, $0.083$ pour $\theta_H=2$, $\approx 0.10$ pour $\theta_H=3$. Plus
les types sont différents, plus le loyer à verser au type haut (et la distorsion du type
bas) sont gros — le résultat qu'on attendait, **mesuré** plutôt que supposé.


## 6. Exercices

### Exercice 1 — Trois types
Étendez le calcul à trois types $\theta \in \{1, 1.5, 2\}$ (probabilités
$1/3$ chacun). Calculez le **loyer d'information** du type du milieu.

*Indice : dans un menu à $n$ types, seul le type le plus bas a un loyer nul ; chaque type
supérieur cumule les écarts de coût avec le type juste en dessous.*


In [6]:
# TODO étudiant — loyer d'information du type du milieu (3 types)
theta_mid = 1.5
# calculez a_H_2B du type milieu puis son loyer U_mid
loyer_type_milieu = None  # TODO étudiant
print("Exercice à compléter : loyer du type milieu =", loyer_type_milieu)

Exercice à compléter : loyer du type milieu = None


### Exercice 2 — Vérifier l'auto-sélection
Vérifiez numériquement que, au second optimum, le **type haut** ne préfère pas le contrat
du type bas (c'est cette contrainte d'auto-sélection, **liante**, qui fixe le loyer).

*Indice : comparez l'utilité du type haut s'il prenait le contrat bas, $t_L - C(\theta_H, a_L)$,
à son utilité sur son propre contrat.*


In [7]:
# TODO étudiant — le type haut préfère-t-il le contrat bas ?
utilite_haut_si_contrat_bas = None  # TODO étudiant : t_L_2B - C(theta_H, a_L_2B)
auto_selection_respectee = None     # TODO étudiant : True si <= utilité sur son propre contrat
print("Exercice à compléter : utilité du haut sur contrat bas =",
      utilite_haut_si_contrat_bas, "-> auto-sélection respectée ?", auto_selection_respectee)


Exercice à compléter : utilité du haut sur contrat bas = None -> auto-sélection respectée ? None


### Exercice 3 — Égalité loyer / distorsion
Pour $\theta_L = 1$, trouvez la valeur de $\theta_H$ pour laquelle le **loyer espéré**
(type haut) et la **distorsion espérée** (type bas) sont **égaux** — le point où les
deux composantes du coût de l'asymétrie se compensent.

*Indice : c'est le $\theta_H$ où `rent_expected == distort_expected`. (Vérifiez-le, ne
l'affirmez pas.)*


In [8]:
# TODO étudiant — theta_H où loyer espéré == distorsion espérée
theta_loyer_dist = None  # TODO étudiant
print("Exercice à compléter : theta_H de l'égalité loyer/distorsion =", theta_loyer_dist)


Exercice à compléter : theta_H de l'égalité loyer/distorsion = None


## 7. Ce que ce carnet établit

- Le **screening** n'est pas sans coût : le type informé (type haut) paie un **loyer
  d'information**, et le type bas est **déformé** (travaille moins que l'optimum). Le
  type haut, lui, reste à son premier optimum — **non-distorsion au sommet**.
- Ce coût se **décompose** de façon exacte en loyer (type haut) + distorsion (type bas) —
  ici loyer $1/18$ + distorsion $1/36$ = coût espéré $1/12$, ce qui n'est **pas** un
  partage 50/50 mais une conséquence des paramètres choisis.
- Le coût **croît convexe** avec l'écart de type : l'asymétrie d'information coûte d'autant
  plus cher que les types sont différents.

C'est le fondement des mécanismes de §16 (conception de mécanismes) : tout contrat qu'on
propose à un agent informé doit payer ce loyer, et le concevoir, c'est minimiser ce coût.
